In [3]:
# Librerias necesarias
import os
import sys
import json
import time
import pandas as pd
from copy import deepcopy
from generacion_pacientes import generar_pacientes
from collections import deque
import parametros as p
from kpis import *
from clases import *

In [4]:
# Cargar los datos
df = pd.read_csv("resultados simulacion/ModeloA_None_T4500_C4208/logs/0.csv")

In [ ]:
# Salidas por ciclo de pacientes en ICU
# 1. Filtrar solo estancias en ICU y ordenar
df_icu = df[df["UNIDAD"] == "ICU"].copy()
df_icu = df_icu.sort_values(["ID", "TI"])

# 2. Obtener siguiente unidad y su tiempo para cada paciente
df_siguiente = df.sort_values(["ID", "TI"]).copy()
df_siguiente["unidad_siguiente"] = df_siguiente.groupby("ID")["UNIDAD"].shift(-1)
df_siguiente["ti_siguiente"] = df_siguiente.groupby("ID")["TI"].shift(-1)

# 3. Unir con ICU para ver su salida
df_icu = df_icu.merge(df_siguiente[["ID", "UNIDAD", "unidad_siguiente", "ti_siguiente"]], 
                      on=["ID", "UNIDAD"], how="left")

# 4. Detectar si salió de ICU (pasó a otra unidad o terminó en ICU)
df_icu_salidas = df_icu[
    df_icu["unidad_siguiente"].notna() | 
    (df_icu["TF"] == df_icu.groupby("ID")["TF"].transform("max"))
]

# 5. Determinar hora de salida
df_icu_salidas["hora_salida"] = df_icu_salidas["ti_siguiente"].fillna(df_icu_salidas["TF"])

# 6. Calcular ciclo (12 horas)
df_icu_salidas["ciclo"] = (df_icu_salidas["hora_salida"] // 12).astype(int)

# 7. Agrupar por ciclo
salidas_por_ciclo = df_icu_salidas.groupby("ciclo").size().reset_index(name="salidas_ICU")

# 8. Mostrar
display(salidas_por_ciclo[(salidas_por_ciclo["ciclo"] > 2000) & salidas_por_ciclo["ciclo"] < 3000].mean())

In [ ]:
# Porcentaje de evoluciones de pacientes que subieron de unidad
nivel_unidades = {"OR": 1, "ICU": 2, "SDU_WARD": 3}

# 2. Filtrar y mapear niveles
df_nivel = df[df["UNIDAD"].isin(nivel_unidades)].copy()
df_nivel["nivel"] = df_nivel["UNIDAD"].map(nivel_unidades)

# 3. Ordenar por tiempo por paciente
df_nivel = df_nivel.sort_values(["ID", "TI"])

# 4. Obtener lista de unidades por paciente
unidades_por_paciente = df_nivel.groupby("ID")["UNIDAD"].agg(list).reset_index()

# 5. Función para detectar subidas
def detectar_subidas(unidades):
    subidas = []
    niveles = [nivel_unidades[u] for u in unidades]
    for i in range(len(niveles) - 1):
        if niveles[i+1] < niveles[i]:  # subió de complejidad
            subidas.append((unidades[i], unidades[i+1]))
    return subidas

# 6. Aplicar y clasificar pacientes
unidades_por_paciente["subidas"] = unidades_por_paciente["UNIDAD"].apply(detectar_subidas)
unidades_por_paciente["subio"] = unidades_por_paciente["subidas"].apply(lambda x: len(x) > 0)

# 7. Porcentajes globales
total_pacientes = len(unidades_por_paciente)
pacientes_subieron = unidades_por_paciente["subio"].sum()
pacientes_no_subieron = total_pacientes - pacientes_subieron

porcentaje_subieron = round(100 * pacientes_subieron / total_pacientes, 2)
porcentaje_no_subieron = round(100 * pacientes_no_subieron / total_pacientes, 2)

print(f"📊 Pacientes que subieron al menos una vez: {porcentaje_subieron}%")
print(f"📉 Pacientes que nunca subieron: {porcentaje_no_subieron}%")

# 8. Proporciones de cada tipo de subida
from collections import Counter
todas_subidas = sum(unidades_por_paciente["subidas"], [])
conteo_subidas = Counter(todas_subidas)

df_subidas = pd.DataFrame(conteo_subidas.items(), columns=["De_A", "conteo"])
df_subidas[["de", "a"]] = pd.DataFrame(df_subidas["De_A"].tolist(), index=df_subidas.index)
df_subidas.drop(columns="De_A", inplace=True)
df_subidas["proporcion_%"] = (df_subidas["conteo"] / df_subidas["conteo"].sum() * 100).round(2)

display(df_subidas)

In [ ]:
# Porcentaje de GRDs y requerimientos iniciales
# 1. Agrupar por ID y quedarte con MS_GRD y requerimiento_inicial únicos por paciente
df_tipos = df.groupby("ID").first()[["MS_GRD", "requerimiento_inicial"]].reset_index()

# 2. Contar ocurrencias de cada tipo de paciente
conteo_tipos = df_tipos.groupby(["MS_GRD", "requerimiento_inicial"]).size().reset_index(name="conteo")

# 3. Calcular el porcentaje
total_pacientes = conteo_tipos["conteo"].sum()
conteo_tipos["porcentaje"] = (conteo_tipos["conteo"] / total_pacientes * 100).round(2)
conteo_tipos
# conteo_tipos[(conteo_tipos["MS_GRD"] < 5) & (conteo_tipos["requerimiento_inicial"] == 1)].sort_values("porcentaje", ascending=False)

In [8]:
# Clase simulacion modificada
class Simulacion: # Revisado, funciona bien

    def __init__(self, T_max, seed, ciclos, modelo = Modelo(), modelo_alternativo = Modelo(), ciclo_de_cambio = 0, pacientes_caso_base = False, log_detallado = False):
        # Asi no vuelvo a crear el archivo de pacientes si ya existe
        """Se empieza generando los pacientes con la semilla de random
        y la cantidad de ciclos que se desean crear pacientes"""
        t0 = time.time()
        self.seed = seed
        self.pacientes_caso_base = pacientes_caso_base
        self.log_detallado = log_detallado
        if self.pacientes_caso_base == False:
            folder_path = "resultados incertidumbre"
            self.file_name = f"{self.seed}_{ciclos}.json"
            file_path = os.path.join(folder_path, self.file_name)
            if os.path.isfile(file_path):
                print(f"Se utiliza archivo existente {self.file_name} de pacientes ({time.time() - t0:.2f} segundos)")
            else:
                generar_pacientes(self.seed, ciclos)
                print(f"Se crea archivo {self.file_name} de pacientes ({time.time() - t0:.2f} segundos)")
        t0 = time.time()
        self.T_max = T_max
        self.pacientes_separados_por_llegada = self.cargar_pacientes_separados_por_llegada()
        print(f"Pacientes separados por llegada cargados ({time.time() - t0:.2f} segundos)")
        t0 = time.time()
        self.ciclos = ciclos
        self.modelo = modelo
        self.modelo_alternativo = modelo_alternativo
        self.ciclo_de_cambio = ciclo_de_cambio
        self.hospital_1 = Hospital(1)
        self.hospital_2 = Hospital(2)
        self.hospital_3 = Hospital(3)
        self.wl = WL([1, 2, 3], [5, 6, 7, 8])
        self.hospitales = { # Para acceder a los hospitales por su id
            0: self.wl,
            p.dict_hospitales["Hospital_1"]: self.hospital_1,
            p.dict_hospitales["Hospital_2"]: self.hospital_2,
            p.dict_hospitales["Hospital_3"]: self.hospital_3
        }
        self.ps = PS([1, 2, 3], [1, 2, 3, 4, 5, 6, 7, 8])
        self.end = END([1, 2, 3], [1, 2, 3, 4, 5, 6, 7, 8])
        self.unidades_termino = { # Para acceder a las unidades de termino por su id
            p.dict_unidades["PS"]: self.ps,
            p.dict_unidades["END"]: self.end
        }
        self.budget = p.budget
        self.tasa_descuento = p.tasa_descuento
        print(f"Clase Simulacion instanciada ({time.time() - t0:.2f} segundos)")

    # Funcion necesaria al momento de instanciar la clase
    def cargar_pacientes_separados_por_llegada(self): # Revisado, funciona bien
        # Cargo los datos necesarios y transformo las llaves nuevamente a int (que se habian vuelto str)
        incertidumbre = {}
        for hospital in range(0,4): # 0 son llegadas a WL
            incertidumbre[hospital] = {}
            for requerimiento in range(1,4):
                incertidumbre[hospital][requerimiento] = {}

        if self.pacientes_caso_base == False:
            with open(f"resultados incertidumbre/{self.file_name}", "r") as file:
                incertidumbre_keys_str = json.load(file)
        else:
            with open("resultados incertidumbre/incertidumbre_base.json", "r") as file:
                incertidumbre_keys_str = json.load(file)

        for hospital in range(0,4): # 0 son llegadas a WL
            for requerimiento in range(1,4):
                for grd in range(1,9):
                    incertidumbre[hospital][requerimiento][grd] = []
                    lista = incertidumbre_keys_str[str(hospital)][str(requerimiento)][str(grd)]
                    if lista != []:
                        for data in lista:
                            arreglado = {
                            'TI': data["TI"],
                            'camino': {1: data['camino']["1"], 2: data['camino']["2"], 3: data['camino']["3"]},
                            'espera': {1: data['espera']["1"], 2: data['espera']["2"], 3: data['espera']["3"]}
                            }
                            if self.pacientes_caso_base:
                                arreglado["decisiones"] = data["decisiones"]
                                arreglado["id"] = data["id"]

                            incertidumbre[hospital][requerimiento][grd].append(arreglado)
                    else:
                        incertidumbre[hospital][requerimiento][grd] = []

        # Se instancian a todos los pacientes a partir de los datos de incertidumbre
        pacientes = {}
        lista_pacientes = []
        # 0 son llegadas a WL
        for hospital in range(0,4): 
            pacientes[hospital] = {}
            for requerimiento in range(1,4):
                pacientes[hospital][requerimiento] = {}
                for grd in range(1,9):
                    pacientes[hospital][requerimiento][grd] = []
                    cantidad_pacientes = len(incertidumbre[hospital][requerimiento][grd])
                    if cantidad_pacientes != 0:
                        for i in range(cantidad_pacientes):
                            paciente = Paciente(hospital, requerimiento, grd, incertidumbre[hospital][requerimiento][grd][i]) # i es el index de la lista
                            pacientes[hospital][requerimiento][grd].append(paciente)
                            lista_pacientes.append(paciente)
                    else:
                        pacientes[hospital][requerimiento][grd] = []

        # Se generan tantas listas como dias con llegadas haya
        pacientes_separados_por_llegada = {}
        for paciente in lista_pacientes:
            ciclo = paciente.ti_inicial
            if ciclo not in pacientes_separados_por_llegada:
                pacientes_separados_por_llegada[ciclo] = []
            pacientes_separados_por_llegada[ciclo].append(paciente)

        return pacientes_separados_por_llegada
    
    # Funciones necesarias al momento de simular
    def agregar_pacientes_ciclo_a_wl(self, ciclo): # Revisado, funciona bien
        pacientes_ciclo = self.pacientes_separados_por_llegada.get(ciclo, [])
        for paciente in pacientes_ciclo:
            # hospital 0 es WL
            if paciente.hospital_llegada == 0:
                self.wl.agregar_paciente(paciente)
            else:
                pass

    def agregar_pacientes_ciclo_a_ed(self, ciclo): # Revisado, funciona bien
        pacientes_ciclo = self.pacientes_separados_por_llegada.get(ciclo, [])
        for paciente in pacientes_ciclo:
            # si hospital es 1, 2 o 3 llegan a ED
            if paciente.hospital_llegada != 0:
                self.hospitales[paciente.hospital_llegada].agregar_paciente(paciente, paciente.unidad_actual)
            else:
                pass

    def sacar_paciente(self, paciente): # Revisado, funciona bien
        # Retorna el paciente que se saca de la unidad, o None si no se pudo sacar
        if paciente.hospital_actual == 0:
            return self.hospitales[paciente.hospital_actual].sacar_paciente(paciente)  
        else:
            return self.hospitales[paciente.hospital_actual].sacar_paciente(paciente, paciente.unidad_actual)

    def agregar_paciente(self, paciente, hospital, unidad): # Revisado, funciona bien
        # Retorna True si se agrega el paciente, False si no
        if unidad not in (p.dict_unidades["PS"], p.dict_unidades["END"]):
            return self.hospitales[hospital].agregar_paciente(paciente, unidad)
        else:
            # No retornan nada porque siempre se agrega el paciente
            self.unidades_termino[unidad].agregar_paciente(paciente)
            return True

    def implementar_decisiones(self, decisiones: list): # Revisado, funciona bien
        """Las decisiones son una lista de diccionarios, cada uno con la siguiente estructura:
        {
            paciente: {"hospital": hospital, "unidad": unidad},
            ...
            paciente: {"hospital": hospital, "unidad": unidad},
        }
        Estas se deben implementar en el orden de la lista, se sacan a todos los pacientes de su
        unidad actual y se cambian a la unidad y hospital que se indica en el diccionario. Luego se 
        pasa al siguiente diccionario y se repite el proceso. Se hace de esta manera porque a veces
        puede ocurrir que dos pacientes tengan que ser cambiados entre si, por lo que los cambios
        deben ser simultaneos por cada diccionario, para sacar a un paciente no necesito saber donde
        esta ya que cada paciente contiene su unidad y hospital actual.
        """
        for decision in decisiones:
            # Saco a todos los pacientes de su unidad actual
            for paciente in decision:
                sacado = self.sacar_paciente(paciente)
                if paciente.esperando == False:
                    print(f"Error no esperando: id: {paciente.id}, h:{paciente.hospital_actual}, u: {paciente.unidad_actual}, caminos: {paciente.camino}, espera: {paciente.espera}")
                    
                if sacado == None:
                    print(f"Error al sacar paciente {paciente.id} de h:{paciente.hospital_actual}, u: {paciente.unidad_actual}")

            # Agrego a todos los pacientes a su nueva unidad
            for paciente, destino in decision.items():
                if self.agregar_paciente(paciente, destino["hospital"], destino["unidad"]):
                    pass
                else:
                    print(f"Error al agregar paciente {paciente.id} a h:{destino['hospital']}, u: {destino['unidad']}")
             
    def actualizar_tiempo(self): # Revisado, funciona bien
        self.hospital_1.actualizar_tiempo()
        self.hospital_2.actualizar_tiempo()
        self.hospital_3.actualizar_tiempo()
        self.wl.actualizar_tiempo()
        # Incrementar el tiempo del sistema
        self.T += 1

    def entregar_log_pacientes_terminados_como_data_frame(self): # Revisado, funciona bien
        t0 = time.time()
        log_completo = []
        for requerimiento in [1, 2, 3]:
            for grd in [1, 2, 3, 4, 5, 6, 7, 8]:
                for paciente in self.end.sub_listas[requerimiento][grd]:
                    contador = 0                    
                    for evento in paciente.log_eventos.copy():
                        contador += 1
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        log_completo.append(evento)
                    contador = 0

                for paciente in self.ps.sub_listas[requerimiento][grd]:
                    for evento in paciente.log_eventos.copy():
                        contador += 1
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        log_completo.append(evento)
                    
        # Convertir la lista de eventos a un DataFrame
        df_log = pd.DataFrame(log_completo)
        # Ordenar el DataFrame por ID y TI y TF
        df_log.sort_values(by=['ID', 'orden'], inplace=True)
        # Resetear el índice
        df_log.reset_index(drop=True, inplace=True)
        print(f"Log de pacientes terminado como DataFrame ({time.time() - t0:.2f} segundos)")
        return df_log

    def entregar_log_detallado_pacientes_terminados_como_data_frame(self): # En proceso
        t0 = time.time()

        def calcular_costo_espera(row):
            drg = row["MS_GRD"]
            los = row["LOS"]
            unidad = row.get("requerimiento_inicial", None)
            ubicacion = row["UBICACIÓN"]
            hospital_nombre = row["HOSPITAL"]
            hospital = p.dict_hospitales.get(hospital_nombre)

            # 1. Paciente en WL_WL (esperando en lista)
            if row["HOSPITAL"] == "WL" and ubicacion == "WL_WL":
                return p.dict_costo_espera_wl[drg][unidad] * los
                
            # 2. Paciente en GA o ED con LOS > 0
            elif row["UNIDAD"] in {"GA", "ED"} and los > 0:
                source = p.dict_costo_espera_ga if row["UNIDAD"] == "GA" else p.dict_costo_espera_ed
                return source[hospital][drg][unidad] * los
                
            # 3. Paciente hospitalizado y bloqueado
            elif row["UNIDAD"] in {"OR", "ICU", "SDU_WARD"} and "->" in ubicacion and los > 0:
                try:
                    origen_str, destino_str = ubicacion.split(" -> ")
                    unidad_actual = "_".join(origen_str.split("_")[2:])
                    unidad_requerida = "_".join(destino_str.split("_")[2:])
                    return p.dict_costo_espera_hospitalizado[hospital][drg][p.dict_unidades[unidad_actual]][p.dict_unidades[unidad_requerida]] * los
                except Exception:
                    return 0  # fallback in case of malformed UBICACION
            return 0

        def parse_hospital_number(hospital_str):
            return int(hospital_str.split("_")[1])

        log_completo = []
        for requerimiento in [1, 2, 3]:
            for grd in [1, 2, 3, 4, 5, 6, 7, 8]:
                pacientes = self.end.sub_listas[requerimiento][grd] + self.ps.sub_listas[requerimiento][grd]
                for paciente in pacientes:
                    contador = 0   
                    # for evento in paciente.log_eventos.copy():
                    tl = paciente.log_eventos.copy()
                    for i in range(len(tl) - 1):
                        evento = tl[i].copy()
                        contador += 1
                        evento["LOS"] = evento["TF"] - evento["TI"]
                        evento["COSTO DER WL"] = 0
                        evento["COSTO DER ED"] = 0
                        evento["COSTO TRASLADO"] = 0
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        evento["COSTO ESPERA"] = calcular_costo_espera(evento)
                        log_completo.append(evento)
                
                        row_current = evento.copy()
                        row_next = tl[i + 1]
                        time_gap = row_current['TF'] < row_next['TI']
                        time_gap_cero = row_current['TF'] == row_next['TI']
                        same_hospital = row_current['HOSPITAL'] == row_next['HOSPITAL']

                        new_row = {
                                'ID': row_current['ID'],
                                'MS_GRD': row_current['MS_GRD'],
                                'UBICACIÓN': f"{row_current['UBICACIÓN']} -> {row_next['UBICACIÓN']}",
                                'TI': row_current['TF'],
                                'TF': row_next['TI'],
                                'LOS': row_next['TI'] - row_current['TF'],
                                'HOSPITAL': row_current['HOSPITAL'],
                                'orden': row_current['orden'] + 0.1,  # Ajustar el orden
                                'requerimiento_inicial': row_current['requerimiento_inicial']
                        }
                        
                        if time_gap:
                            new_row.update({
                                'UNIDAD': row_current['UNIDAD']
                            })
                            
                            costo_espera = calcular_costo_espera(new_row)

                            new_row.update({
                                "COSTO DER WL": 0,
                                "COSTO DER ED": 0,
                                "COSTO TRASLADO": 0,
                                "COSTO ESPERA": costo_espera
                            })

                            log_completo.append(new_row)
                        
                        elif time_gap_cero and not same_hospital:
                            x = new_row.copy()
                            traslados = {
                                'Hospital_1_ED -> Hospital_2_ED',
                                'Hospital_1_ED -> Hospital_3_ED',
                                'Hospital_2_ED -> Hospital_1_ED',
                                'Hospital_2_ED -> Hospital_3_ED',
                                'Hospital_3_ED -> Hospital_1_ED',
                                'Hospital_3_ED -> Hospital_2_ED'
                            }
                            costo_der_wl = p.dict_costo_derivar_wl[x["MS_GRD"]][x["requerimiento_inicial"]] if x["UBICACIÓN"] == "WL_WL -> PS_PS" else 0
                            costo_der_ed = p.dict_costo_derivar_ed[p.dict_hospitales[x["HOSPITAL"]]][x["MS_GRD"]][x["requerimiento_inicial"]] if x["UBICACIÓN"] == f"{x['HOSPITAL']}_ED -> PS_PS" else 0
                            costo_traslado = (p.dict_costo_traslado[parse_hospital_number(x["UBICACIÓN"].split(" -> ")[0])]
                                              [parse_hospital_number(x["UBICACIÓN"].split(" -> ")[1])][x["MS_GRD"]][x["requerimiento_inicial"]]
                                            if x["UBICACIÓN"] in traslados else 0)

                            new_row.update({
                                'UNIDAD': "En movimiento",
                                "COSTO DER WL": costo_der_wl,
                                "COSTO DER ED": costo_der_ed,
                                "COSTO TRASLADO": costo_traslado,
                                "COSTO ESPERA": 0
                            })
                    
                            log_completo.append(new_row)
                    
                    evento = tl[-1].copy()
                    contador += 1
                    evento["LOS"] = evento["TF"] - evento["TI"]
                    evento["COSTO DER WL"] = 0
                    evento["COSTO DER ED"] = 0
                    evento["COSTO TRASLADO"] = 0
                    evento["COSTO ESPERA"] = 0
                    evento["orden"] = contador
                    evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                    log_completo.append(evento)

        # Convertir la lista de eventos a un DataFrame
        df_log = pd.DataFrame(log_completo)
        # Ordenar el DataFrame por ID y TI y TF
        df_log.sort_values(by=['ID', 'orden'], inplace=True)
        df_log = df_log[['ID', 'MS_GRD', 'UBICACIÓN', 'TI', 'TF', 'LOS', 'HOSPITAL', 'UNIDAD', 'requerimiento_inicial', 'COSTO DER WL', 'COSTO DER ED', 'COSTO TRASLADO', 'COSTO ESPERA']]
        # Resetear el índice
        df_log.reset_index(drop=True, inplace=True)
        df_log[["TI", "TF", "LOS"]] = df_log[["TI", "TF", "LOS"]] * 12
        print(f"Log de pacientes terminado como DataFrame ({time.time() - t0:.2f} segundos)")
        return df_log

    def simular(self):
        t0 = time.time()
        self.T = 1  # Inicializa el tiempo del sistema en 1

        # Bucle principal de simulación
        while self.T <= self.T_max:
            if self.T % 1000 == 0:
                print(f"\nSimulando ciclo {self.T} de {self.T_max}")
            
            # Esto solo ocurre cuando quiero cambiar de modelo entremedio de la simulacion T!= 0
            if self.T == self.ciclo_de_cambio:
                self.modelo = self.modelo_alternativo
            
            # Agregar pacientes a la WL según el ciclo actual
            self.agregar_pacientes_ciclo_a_wl(self.T)

            # Agregar pacientes a las ED según el ciclo actual
            self.agregar_pacientes_ciclo_a_ed(self.T)

            # Entrego el estado de la simulación al modelo para tomar decisiones (argumento self)
            if self.T == 2500:
                return self.modelo.devolverme(self)

            decisiones = self.modelo.tomar_decisiones(self)

            # Se implementan todas las decisiones del modelo
            self.implementar_decisiones(decisiones)

            # Actualizar el tiempo de cada unidad y paciente
            self.actualizar_tiempo()
            
        # Al finalizar la simulación, se entregan los logs de los pacientes terminados
        print(f"Simulación finalizada en ({time.time() - t0:.2f} segundos)")
        if self.log_detallado:
            return self.entregar_log_detallado_pacientes_terminados_como_data_frame()
        else:
            return self.entregar_log_pacientes_terminados_como_data_frame()
        
    def __str__(self):
        pass



In [9]:
class ModeloB(ModeloA):
    """Espera hasta que la Wl se llene con mas de 1000 personas, de ahi espera 100 ciclos mas
    y luego empieza a atender pacientes de la WL directamente, por lo que esta empieza a bajar"""
    def __init__(self):
        super().__init__()
    
    def devolverme(self, simulacion):
        # Reinicio las variables
        self.decisiones = []
        self.expulsados_wl_del_ciclo_actual = []
        self.actual = self.actual_vacio.copy()
        self.budget = p.budget

        # Copio localmente las ocupaciones de cada hospital
        self.cargar_ciclo(simulacion)
        # Reviso si hay pacientes que estoy obligado a sacar de WL (no implementado todavia)
        self.agregar_pacientes_obligatorio_a_ga() 
        # Primero reviso si hay pacientes que deben ser dados de alta
        self.dar_de_alta()

        return self.actual

In [ ]:

clase_modelo_base = ModeloB
T_max=4500
ciclos=4208
num_simulaciones=1
seed_inicial=0
clase_modelo_alternativo=None
ciclo_de_cambio=0
pacientes_caso_base=False
log_detallado=True

# Nombres base y alternativo para la carpeta principal
nombre_base = clase_modelo_base.__name__
nombre_alternativo = clase_modelo_alternativo.__name__ if clase_modelo_alternativo else "None"
nombre_carpeta = f"{nombre_base}_{nombre_alternativo}_T{T_max}_C{ciclos}"

# Crear carpeta principal y subcarpetas
base_dir = os.path.join("resultados simulacion", nombre_carpeta)
logs_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
kpis_dir = os.path.join(base_dir, "kpis")
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(kpis_dir, exist_ok=True)

for i in range(num_simulaciones):
    seed = seed_inicial + i
    print(f"\n⏳ Simulación {i+1}/{num_simulaciones} con seed={seed}")

    # Reiniciar contador global
    Paciente.CONTADOR_ID = 1

    # Instanciar modelos
    modelo = clase_modelo_base()
    alternativo = clase_modelo_alternativo() if clase_modelo_alternativo else None

    # Ejecutar simulación
    simu = Simulacion(
        T_max, seed, ciclos,
        modelo=modelo,
        modelo_alternativo=alternativo,
        pacientes_caso_base=pacientes_caso_base,
        ciclo_de_cambio=ciclo_de_cambio,
        log_detallado=log_detallado
    ) 

    estado = simu.simular()

In [11]:
actual = deepcopy(estado)

In [ ]:
def pretty_print_dict(d, indent=0):
    for key, value in d.items():
        prefix = " " * indent + f"{key}: "
        if isinstance(value, dict):
            print(prefix)
            pretty_print_dict(value, indent + 4)
        elif isinstance(value, list):
            print(prefix + f"list of length {len(value)}")
        elif isinstance(value, deque):
            print(prefix + f"deque of length {len(value)}")
        else:
            print(prefix + str(value))

pretty_print_dict(actual)

In [13]:
hoy = 0
antes = 0
for req in range(1, 4):
    for grd in range(5, 9):
        for paciente in actual["WL_sub_deques"][req][grd]:
            if paciente.ti_evento_actual == 2500:
                hoy += 1
            else:
                antes += 1

print(f"Pacientes en WL hoy: {hoy}, antes: {antes}, total: {hoy + antes}")

Pacientes en WL hoy: 18, antes: 19, total: 37


In [235]:
class ModeloProactivo(ModeloA):
    """Modelo que implementa una estrategia proactiva para manejar el flujo de pacientes en el hospital."""
    def __init__(self):
        super().__init__()
        demanda_ed, demanda_wl = self.generar_demanda_ed_wl()
        self.demanda_ed = demanda_ed
        self.demanda_wl = demanda_wl
    
    # Generar demanda promedio a partir de llegadas estimadas
    def generar_demanda_ed_wl(self):
    
        def cargar_llegadas_y_estimar_esperados(path_relativo="../1. codigo analisis/resultados incertidumbre/llegadas.json"):
            ruta = os.path.abspath(path_relativo)
            
            with open(ruta, "r") as file:
                datos_llegadas = json.load(file)

            esperados = {}

            for hospital_id, requerimientos in datos_llegadas.items():
                hospital_id = int(hospital_id)
                for req_id, grds in requerimientos.items():
                    req_id = int(req_id)
                    for grd_id, contenido in grds.items():
                        grd_id = int(grd_id)
                        if "final_kde_pmf" in contenido:
                            pmf = contenido["final_kde_pmf"]
                            esperado = sum(i * p for i, p in enumerate(pmf))
                            clave = (hospital_id, req_id, grd_id)
                            esperados[clave] = esperado

            return esperados

        def generar_bloque_deterministico_preciso(promedio_deseado):
            """
            Genera un bloque determinístico con longitud igual a 10^donde d = cantidad de decimales de promedio_deseado.
            Esto asegura una representación exacta del promedio si los decimales son finitos.

            Retorna: lista de enteros que alternan entre base y base+1, con promedio igual al deseado.
            """
            # Obtener número de decimales del número como string
            partes = str(promedio_deseado).split(".")
            decimales = len(partes[1]) if len(partes) == 2 else 0
            largo_bloque = 10 ** decimales if decimales > 0 else 1

            base = int(float(promedio_deseado))
            extra = promedio_deseado - base
            n_altos = round(extra * largo_bloque)
            n_bajos = largo_bloque - n_altos

            bloque = [base] * largo_bloque

            if n_altos > 0:
                paso = largo_bloque / n_altos
                for i in range(n_altos):
                    idx = round(i * paso)
                    if idx >= largo_bloque:
                        idx = largo_bloque - 1
                    while bloque[idx] == base + 1:
                        idx = (idx + 1) % largo_bloque
                    bloque[idx] = base + 1

            if extra >= 0.5:
                bloque = bloque[::-1]  # Invertir el bloque si el promedio es mayor o igual a 0.5

            return bloque

        def transformar_df_a_dict(df):
            """
            Transforma un DataFrame con columnas:
            ['hospital_id', 'requerimiento', 'total_esperado', 'demanda_ciclo']
            a un diccionario anidado {hospital_id: {requerimiento: {total_esperado, llegadas}}}
            """
            resultado = {}

            for _, row in df.iterrows():
                hosp = row["hospital_id"]
                req = row["requerimiento"]
                esperado = row["total_esperado"]
                llegadas = row["demanda_ciclo"]

                if hosp not in resultado:
                    resultado[hosp] = {}

                resultado[hosp][req] = {
                    "total_esperado": esperado,
                    "d_ciclo": llegadas
                }

            return resultado
        
        llegadas = cargar_llegadas_y_estimar_esperados()
        df_llegadas = pd.DataFrame([
            {"hospital_id": int(k[0]), "requerimiento": int(k[1]), "GRD": int(k[2]), "esperado": round(v,2)}
            for k, v in llegadas.items()
        ])

        # Demanda de ED
        df_llegadas_ed = df_llegadas[df_llegadas["hospital_id"] != 0].copy()
        df_llegadas_ed["costo"] = df_llegadas_ed.apply(lambda row: p.dict_costo_derivar_ed[row["hospital_id"]][row["GRD"]][row["requerimiento"]], axis=1)
        df_llegadas_ed["costo_esperado"] = round(df_llegadas_ed["esperado"] * df_llegadas_ed["costo"],2)
        ed_mod = df_llegadas_ed.groupby(["hospital_id", "requerimiento"]).agg(
            total_esperado=("esperado", "sum"),
            total_costo_esperado=("costo_esperado", "sum")
        ).reset_index().sort_values(by=["hospital_id", "requerimiento"])
        ed_mod["demanda_ciclo"] = ed_mod.apply(lambda row: generar_bloque_deterministico_preciso(row["total_esperado"]), axis=1)
        demanda_ed = transformar_df_a_dict(ed_mod)

        # Demanda de WL
        df_demanda_wl = df_llegadas[df_llegadas["hospital_id"] == 0].copy()
        wl_mod = df_demanda_wl.groupby(["hospital_id", "requerimiento"]).agg(total_esperado=("esperado", "sum")
        ).reset_index().sort_values(by=["hospital_id", "requerimiento"])
        wl_mod["demanda_ciclo"] = wl_mod.apply(lambda row: generar_bloque_deterministico_preciso(row["total_esperado"]), axis=1)
        demanda_wl = transformar_df_a_dict(wl_mod)
        
        return demanda_ed, demanda_wl
    
    # Decidir la demanda de camas por ciclo, limite inferior o superior
    def demandas_ed_por_ciclo(self, ciclo):
            """
            Retorna un diccionario con los valores de camas a asignar en un ciclo específico
            para todas las combinaciones de hospital y requerimiento.

            Parámetros:
            - ciclo (int): número de ciclo
            - dict_camas (dict): diccionario con estructura:
                {hospital_id: {requerimiento: {"d_ciclo": [valores_por_ciclo]}}}

            Retorna:
            - dict: {hospital_id: {requerimiento: valor_para_ciclo}}
            """
            dict_camas = self.demandas_ed
            resultado = {}
            for hospital_id, requerimientos in dict_camas.items():
                resultado[hospital_id] = {}
                for requerimiento, datos in requerimientos.items():
                    d_ciclo = datos["d_ciclo"]
                    valor = d_ciclo[ciclo % len(d_ciclo)]
                    resultado[hospital_id][requerimiento] = valor
            return resultado

    

In [236]:
modelo = ModeloProactivo()

{1: {1: {'total_esperado': 2.4, 'd_ciclo': [3, 2, 3, 2, 2, 3, 2, 2, 3, 2]},
  2: {'total_esperado': 2.66,
   'd_ciclo': [2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3]},
  3: {'total_esperado': 1.6, 'd_ciclo': [1, 2, 2, 1, 2, 1, 2, 2, 1, 2]}},
 2: {1: {'total_esperado': 2.32,
   'd_ciclo': [3,
    2,
    2,
    3,
    2,
    2,
    3,
    2,
    2,
